# Testing Notebook for BGR Fisher Analysis

## Goal

This notebook tests the PhenomXHM waveform with a -1PN EdGB and a 2PN dCS phase correction. The models used below are `PhenomXHM_EdGB()`, `PhenomXHM_dCS()`, and `PhenomXHM_MG()` whose extra Fisher parameter are `squared_alpha_EdGB_km4 = alpha_EdGB^2` and `squared_alpha_dCS_km4 = alpha_dCS^2` (both in km$^4$), and `inv_lambda_g2_km2 = 1/lambda_g^2` (in km$^2$), respectively. The relative inclusion in the original GWJulia can be found in the $\texttt{.jl}$ file carrying the the same waveform names. 

## Current code organization

* `PhenomXHM_EdGB.jl` contains the EdGB-specific mapping from `squared_alpha_EdGB_km4` to the generic TIGER/ppE coefficient `delta_phi_minus2` (for a -1PN addition). The expression for the correction is given by: 

$$\delta \phi_{-2} = \frac{128.0}{3.0} \ \beta_\mathrm{EdGB} \ \eta^{(-2 / 5)}$$
 
where $\beta_\mathrm{EdGB}$ is the expression found in Eq. (4) of [arXiv:1905.00870v3](https://arxiv.org/abs/1905.00870v3).

* `PhenomXHM_dCS.jl` contains the dCS-specific mapping from `squared_alpha_dCS_km4` to the generic TIGER/ppE coefficient `delta_phi_plus4` (for a 2PN addition). The expression for the correction is given by: 

$$\delta \phi_{+4} = \frac{128.0}{3.0} \ \beta_\mathrm{dCS} \ \frac{\eta^{(4 / 5)}}{\phi_\mathrm{GR, 4}}$$
 
where $\beta_\mathrm{dCS}$ is the expression found in Eq. (3) of [arXiv:1905.00870v3](https://arxiv.org/abs/1905.00870v3) and $\phi_\mathrm{GR, 4}$ is the original phase in GR.

* `PhenomXHM_MG.jl` contains the MG-specific mapping from `inv_lambda_g2_km2` to the generic TIGER/ppE coefficient `delta_phi_plus2` (for a 1PN addition). It also features a function to evaluate redshift $z$ from a given luminosity distance $d_L$. The expression for the correction is given by: 

$$\delta \phi_{+2} = - \frac{128.0}{3.0} \ \beta_\mathrm{MG} \ \frac{\eta^{(2 / 5)}}{\phi_\mathrm{GR, 2}}$$
 
where $\beta_\mathrm{MG}$ is the expression found in Eq. (3.8c, 3.9) of [arXiv:gr-qc/9709011](https://arxiv.org/pdf/gr-qc/9709011) and $\phi_\mathrm{GR, 4}$ is the original phase in GR.

* `PhenomXHM_TIGER_spinless(-1.0, 1.0 or 2.0)` (-1.0, 1.0 and 2.0 indicate the three possible corrections) is available as the generic model where the extra parameters are directly `delta_phi_minus2`, `delta_phi_plus2` and `delta_phi_plus4`. The latter are the ones called in the methods to return the polarizations hphc. 

* `PhenomXHM.jl` and `ConnectionFunctionsXAS.jl` only carry the generic -1PN, 1PN and 2PN phase plumbing, through the `PhenomXHM_TIGER_spinless(-1.0, 1.0 or 2.0)` structure.


In [1]:
using GWInference
using LinearAlgebra
using DSP
using Statistics

In [25]:
# GW170608-like, strongest/cleanest EdGB test (from paper at least)
m1 = 11.0
m2 = 7.6
mc = (m1*m2)^(3/5) / (m1 + m2)^(1/5)
eta = m1*m2 / (m1 + m2)^2

chi_eff = 0.03
chi1 = chi_eff * (m1 + m2) / m1
chi2 = 0.0

dL = 0.32  # Gpc
squared_alpha_EdGB_km4 = 2.0^4 # from reference paper cited treshold
squared_alpha_dCS_km4 = 5.0^4 # the bounds are too large for the other
inv_lambda_g2_km2 = 1e-30 # from paper but not sure 

theta = 1.0
phi = 2.0
iota = 0.7
psi = 0.4
tcoal = 0.0
phiCoal = 0.0

0.0

## Test 1

Test whether the waveform with $o1=0.0$ is the same as the one without passing the BGR factor.

In [26]:
f = collect(range(10.0, 512.0, length=2000))

# std GR model
hp_gr, hc_gr = hphc(PhenomXHM(), f, mc, eta, chi1, chi2, dL, iota)

# BGR model with sqrt_alpha_EdGB_km = 0.0 --> means zero added effect in o1 parameter
hp_0, hc_0 = hphc(
    PhenomXHM_EdGB(),
    f, mc, eta, chi1, chi2, dL, iota, 0.0
)

# BGR model with sqrt_alpha_dCS_km = 0.0 --> means zero added effect in o1 parameter
hp_0_1, hc_0_1 = hphc(
    PhenomXHM_dCS(),
    f, mc, eta, chi1, chi2, dL, iota, 0.0
)

# BGR model with inv_lambda_g2_km2 = 0.0 --> means zero added effect in o1 parameter
hp_0_2, hc_0_2 = hphc(
    PhenomXHM_MG(),
    f, mc, eta, chi1, chi2, dL, iota, 0.0
)

println("GR vs BGR(alpha_EdGB_km^2=0): ", maximum(abs.(hp_gr .- hp_0)))
println("GR vs BGR(alpha_dCS_km^2=0): ", maximum(abs.(hp_gr .- hp_0_1)))
println("GR vs BGR(alpha_MG_km^2=0): ", maximum(abs.(hp_gr .- hp_0_2)))

GR vs BGR(alpha_EdGB_km^2=0): 0.0
GR vs BGR(alpha_dCS_km^2=0): 0.0
GR vs BGR(alpha_MG_km^2=0): 0.0


## Test 2

Test whether the waveform with a nonzero EdGB/dCS coupling is different from GR and finite. The diagnostic value is intentionally not tiny, otherwise the difference is below floating-point precision.

In [27]:
# BGR model with sqrt_alpha_EdGB_km = 0.3

hp_bgr, hc_bgr = hphc(
    PhenomXHM_EdGB(),
    f, mc, eta, chi1, chi2, dL, iota, 0.3
)

hp_bgr_1, hc_bgr_1= hphc(
    PhenomXHM_dCS(),
    f, mc, eta, chi1, chi2, dL, iota, 0.3
)

hp_bgr_2, hc_bgr_2= hphc(
    PhenomXHM_MG(),
    f, mc, eta, chi1, chi2, dL, iota, 0.3
)

println("GR vs BGR(alpha_EdGB_km^2=0.3): ", maximum(abs.(hp_gr .- hp_bgr)))
println("finite? ", all(isfinite, real.(hp_bgr)), " ", all(isfinite, imag.(hp_bgr)))
println("GR vs BGR(alpha_dCS_km^2=0.3): ", maximum(abs.(hp_gr .- hp_bgr_1)))
println("finite? ", all(isfinite, real.(hp_bgr_1)), " ", all(isfinite, imag.(hp_bgr_1)))
println("GR vs BGR(inv_lambda_g2_km2=0.3): ", maximum(abs.(hp_gr .- hp_bgr_2)))
println("finite? ", all(isfinite, real.(hp_bgr_2)), " ", all(isfinite, imag.(hp_bgr_2)))

GR vs BGR(alpha_EdGB_km^2=0.3): 1.8396814037166097e-23
finite? true true
GR vs BGR(alpha_dCS_km^2=0.3): 1.1025552176733203e-30
finite? true true
GR vs BGR(inv_lambda_g2_km2=0.3): 1.338164857848849e-22
finite? true true


## Test 3

Check the expected EdGB scaling (only that). Since the parameter is simple the square of `alpha_EdGB_km2`, namely `squared_alpha_EdGB_km4`, the phase correction scales approximately as `alpha_EdGB_km2^2`, so doubling the parameter should give a ratio close to 2 for small values:

In [28]:
hp_a, _ = hphc(PhenomXHM_EdGB(), f, mc, eta, chi1, chi2, dL, iota, 0.3)
hp_b, _ = hphc(PhenomXHM_EdGB(), f, mc, eta, chi1, chi2, dL, iota, 0.6)

dh_a = hp_a .- hp_gr
dh_b = hp_b .- hp_gr

println("quadratic scaling ratio: ", norm(dh_b) / norm(dh_a))

quadratic scaling ratio: 1.9294894231860915


## Test 4: Fisher analysis

Checks Fisher on waveforms other than mine, and on mine. Then calculate the SNR and covariance to extract errors. 

In [29]:
FisherMatrix(
    PhenomD(),
    CE1Id,
    mc, eta, chi1, chi2, dL,
    theta, phi, iota, psi, tcoal, phiCoal;
    res=100,
    fmin=10.0,
    fmax=512.0,
    rho_thres=nothing
);

println("Fisher matrix calculated")

Fisher matrix calculated


In [30]:
model = PhenomXHM_EdGB()

# find the Fisher matrix for the BGR model with sqrt_alpha_EdGB_km = 0.3

F= FisherMatrix(
    model,
    ETS,
    mc,
    eta,
    chi1,
    chi2,
    dL,
    theta,
    phi,
    iota,
    psi,
    tcoal,
    phiCoal,
    squared_alpha_EdGB_km4;
    res=300,
    fmin=10.0,
    fmax=512.0,
    rho_thres=nothing,
    #return_SNR=true
);

println("Fisher matrix calculated")

Fisher matrix calculated


In [31]:
model_1 = PhenomXHM_dCS()

# find the Fisher matrix for the BGR model with sqrt_alpha_EdGB_km = 0.3

F_1 = FisherMatrix(
    model_1,
    ETS,
    mc,
    eta,
    chi1,
    chi2,
    dL,
    theta,
    phi,
    iota,
    psi,
    tcoal,
    phiCoal,
    squared_alpha_dCS_km4;
    res=300,
    fmin=10.0,
    fmax=512.0,
    rho_thres=nothing,
    #return_SNR=true
);

println("Fisher matrix calculated")

Fisher matrix calculated


In [32]:
model_2 = PhenomXHM_MG()

# find the Fisher matrix for the BGR model with sqrt_alpha_EdGB_km = 0.3

F_2 = FisherMatrix(
    model_2,
    ETS,
    mc,
    eta,
    chi1,
    chi2,
    dL,
    theta,
    phi,
    iota,
    psi,
    tcoal,
    phiCoal,
    inv_lambda_g2_km2;
    res=300,
    fmin=10.0,
    fmax=512.0,
    rho_thres=nothing,
    #return_SNR=true
);

println("Fisher matrix calculated")

Fisher matrix calculated


In [33]:
println("\n==================== EdGB model ====================")
println("Size          : ", size(F))
println("Symmetry error: ", maximum(abs.(F .- F')))
println("Is symmetric? : ", maximum(abs.(F .- F')) ≈ 0)
println("Is finite?    : ", all(isfinite, F))
println("Diagonal      : ", diag(F))

println("\n==================== dCS model ====================")
println("Size          : ", size(F_1))
println("Symmetry error: ", maximum(abs.(F_1 .- F_1')))
println("Is symmetric? : ", maximum(abs.(F_1 .- F_1')) ≈ 0)
println("Is finite?    : ", all(isfinite, F_1))
println("Diagonal      : ", diag(F_1))

println("\n==================== MG model ====================")
println("Size          : ", size(F_2))
println("Symmetry error: ", maximum(abs.(F_2 .- F_2')))
println("Is symmetric? : ", maximum(abs.(F_2 .- F_2')) ≈ 0)
println("Is finite?    : ", all(isfinite, F_2))
println("Diagonal      : ", diag(F_2))


println("=================================================\n")


==================== EdGB model ====================
Size          : (12, 12)
Symmetry error: 0.0
Is symmetric? : true
Is finite?    : true
Diagonal      : [6.47310840129709e9, 7.607690825098734e11, 3.01461403685834e8, 1.1402253345918623e8, 688856.5165688619, 5.761925508488078e6, 3.647627615670504e6, 36844.688000827904, 278219.8875218958, 2.5900268130172253e10, 70538.90729665147, 221597.0106659282]

==================== dCS model ====================
Size          : (12, 12)
Symmetry error: 0.0
Is symmetric? : true
Is finite?    : true
Diagonal      : [5.568132577523739e9, 4.77484048203667e9, 2.906189753669409e8, 1.1431390422279371e8, 691706.7192888592, 5.892789668427536e6, 3.7515234114610613e6, 37341.00033463541, 279528.0287648408, 2.654974177350294e10, 70830.76805517926, 2.9154389336181153e-9]

==================== MG model ====================
Size          : (12, 12)
Symmetry error: 0.0
Is symmetric? : true
Is finite?    : true
Diagonal      : [5.568121410552159e9, 4.7747148287670

In [35]:
snr = SNR(
    model,
    ETS,
    mc, eta, chi1, chi2, dL,
    theta, phi, iota, psi, tcoal,
    squared_alpha_EdGB_km4;
    res=100,
    fmin=10.0,
    fmax=512.0
)

snr_1 = SNR(
    model_1,
    ETS,
    mc, eta, chi1, chi2, dL,
    theta, phi, iota, psi, tcoal,
    squared_alpha_dCS_km4;
    res=100,
    fmin=10.0,
    fmax=512.0
)

snr_2 = SNR(
    model_2,
    ETS,
    mc, eta, chi1, chi2, dL,
    theta, phi, iota, psi, tcoal,
    inv_lambda_g2_km2;
    res=100,
    fmin=10.0,
    fmax=512.0
)

println("\n==================== EdGB model ====================")
println("Signal SNR : ", snr)

println("==================== dCS model ====================")
println("Signal SNR : ", snr_1)

println("==================== MG model ====================")
println("Signal SNR : ", snr_2)

println("=================================================\n")


==================== EdGB model ====================
Signal SNR : 264.87156353962865
==================== dCS model ====================
Signal SNR : 263.66673449218456
==================== MG model ====================
Signal SNR : 263.6667931343359



In [36]:
mycovariance = CovMatrix(F)
mycovariance_1 = CovMatrix(F_1)
mycovariance_2 = CovMatrix(F_2);

Inversion successful
Inversion error: 1.2615930501194376e-5

Inversion error before normalization: 1.3689277905123416
Inversion error after normalization: 0.059917076004526414
Try to invert with 128 bit precision
Inversion successful
Inversion error: 4.479021123584763e-25

Inversion error before normalization: 1.3687157731342607e18
Inversion error after normalization: 1.2519110816001718e18
Try to invert with 128 bit precision
Inversion successful
Inversion error: 3.6058121163880945e-7



In [37]:
# we can now calculate the errors on the parameters

println("\n==================== EdGB model ====================")

myerrors = Errors(mycovariance)
parameters_string = ["mc", "η", "χ_1", "χ_2", "dL", "θ", "ϕ", "ι", "ψ", "tcoal", "Φ_coal", "α_EdGB^2 [km^4]"]

for i in eachindex(myerrors)
    println("The error on $(parameters_string[i]) is $(myerrors[i])")
end

println("=================================================\n")

# second model 

println("==================== dCS model ====================")

myerrors_1 = Errors(mycovariance_1)
parameters_string = ["mc", "η", "χ_1", "χ_2", "dL", "θ", "ϕ", "ι", "ψ", "tcoal", "Φ_coal", "α_dCS^2 [km^4]"]

for i in eachindex(myerrors_1)
    println("The error on $(parameters_string[i]) is $(myerrors_1[i])")
end

println("=================================================\n")

# third model

println("==================== MG model ====================")

myerrors_2 = Errors(mycovariance_2)
parameters_string = ["mc", "η", "χ_1", "χ_2", "dL", "θ", "ϕ", "ι", "ψ", "tcoal", "Φ_coal", "1/λ_MG^2 [km^2]"]

for i in eachindex(myerrors_2)
    println("The error on $(parameters_string[i]) is $(myerrors_2[i])")
end

println("=================================================\n")


==================== EdGB model ====================
The error on mc is 0.0007696442401757795
The error on η is 0.0029038707945745045
The error on χ_1 is 0.1003468533956865
The error on χ_2 is 0.14883835420844455
The error on dL is 0.0425703835370031
The error on θ is 0.022756737373385936
The error on ϕ is 0.015209600953509044
The error on ι is 0.12366258758814076
The error on ψ is 0.9157239431021749
The error on tcoal is 0.0005029375567345394
The error on Φ_coal is 1.8210646806773663
The error on α_EdGB^2 [km^4] is 5.827633885056658

==================== dCS model ====================
The error on mc is 0.0005091164749059186
The error on η is 0.002876598156116535
The error on χ_1 is 0.11775207579558977
The error on χ_2 is 0.17769828414683422
The error on dL is 0.042178116728224864
The error on θ is 0.024564553231733063
The error on ϕ is 0.0183598735709513
The error on ι is 0.1260886115711181
The error on ψ is 1.0308212374952392
The error on tcoal is 0.0005712830699108431
The error on

In [39]:
# find errors on alpha_EdGB, which is the param in the paper we are interested in:

sigma_alpha_squared = myerrors[end]
sigma_sqrt_alpha = sigma_alpha_squared^(1.0 / 4.0)

println("The error on sqrt(α_EdGB^2) is $(sigma_sqrt_alpha)")

# find errors on alpha_dCS, which is the param in the paper we are interested in:

sigma_alpha_squared = myerrors_1[end]
sigma_sqrt_alpha_1 = sigma_alpha_squared^(1.0 / 4.0)

println("The error on sqrt(α_dCS^2) is $(sigma_sqrt_alpha_1)")


# find errors on lambda_MG, which is the param in the paper we are interested in:

lambda_MG_2 = 1 / myerrors_2[end]
sigma_sqrt_lambda_MG = lambda_MG_2^(1. / 2.)

println("The error on sqrt(λ_MG) is $(sigma_sqrt_lambda_MG)")

The error on sqrt(α_EdGB^2) is 1.5537211047881674
The error on sqrt(α_dCS^2) is 34.88542133348915
The error on sqrt(λ_MG) is 1.5753231148277658e13


It's important to notice that I am studying the Fisher matrix for the squared value of the parameters alpha in a way that the derivative with respect to such parameter will be linear and not singular for small values of such parameter. 